# Volatility Environment Phase 4 — R2 Deployment Validation

Current status and numerical results are shown from the saved CSVs below.
Fixed R2=.50/.70/.90/1.10/1.30%; WorstDay must be > -20%. No live changes.
A is theoretical Monday06 Balance; repo EA uses Monday00 key and first-call Equity/Balance snapshot.


In [ ]:
from pathlib import Path
import subprocess, sys, json, tempfile
import pandas as pd
REPO = Path('/content/time-entry-portfolio-lab-phase4')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','research/volatility-environment-phase4-deployment-validation','https://github.com/TR-KJ/time-entry-portfolio-lab.git',str(REPO)],check=True)
SAVED = REPO/'results/volatility_phase4'
record = pd.read_csv(SAVED/'volatility_phase4_run_record.csv').iloc[0]
IMPLEMENTATION_SHA = record.ImplementationSHA
print('Plan:', record.PlanSHA, 'Implementation:', IMPLEMENTATION_SHA, 'Status:', record.Status)


In [ ]:
for name in ['27strategy_money_summary','period_reset_summary','robustness_summary','deployment_feasibility','decision']:
    print(name)
    display(pd.read_csv(SAVED/('volatility_phase4_'+name+'.csv')))


## Regenerate into /content
Supply the original hash-matching baseline and JSON list of 56 original M1 paths. Missing files produce explicit NOT_RUN, never estimated results. Complete real-data validation and current SET/spec audit are still required before candidacy.


In [ ]:
RUN_REGENERATION = False
BASELINE = None  # original daily_stop_baseline_trades.csv
INPUT_PATHS_JSON = None  # JSON list of original M1 file paths
PHASE2_FULL = None  # optional published-hash-matching full assignments CSV
OUTPUT = None
if RUN_REGENERATION:
    CODE = Path(tempfile.mkdtemp(prefix='phase4_code_',dir='/content'))
    subprocess.run(['git','clone',str(REPO),str(CODE)],check=True)
    subprocess.run(['git','checkout',IMPLEMENTATION_SHA],cwd=CODE,check=True)
    OUTPUT = Path(tempfile.mkdtemp(prefix='volatility_phase4_',dir='/content'))
    command = [sys.executable,str(CODE/'src/research/volatility_phase4.py'),'--implementation-sha',IMPLEMENTATION_SHA,'--output-dir',str(OUTPUT)]
    for flag,value in [('--baseline',BASELINE),('--input-paths',INPUT_PATHS_JSON),('--phase2-full',PHASE2_FULL)]:
        if value: command += [flag,str(value)]
    subprocess.run(command,check=True)
    if BASELINE and INPUT_PATHS_JSON:
        SECOND = Path(tempfile.mkdtemp(prefix='volatility_phase4_repeat_',dir='/content'))
        command2 = command.copy()
        command2[command2.index('--output-dir')+1] = str(SECOND)
        subprocess.run(command2,check=True)
        subprocess.run([sys.executable,str(CODE/'tests/finalize_volatility_phase4.py'),'--first',str(OUTPUT),'--second',str(SECOND)],check=True)
    display(pd.read_csv(OUTPUT/'volatility_phase4_decision.csv'))


In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    if OUTPUT is None: raise RuntimeError('Run regeneration first')
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive')/OUTPUT.name
    shutil.copytree(OUTPUT,destination)
